# PPO θ.5 — 1M step BEST-PRACTICE training (Day 5+)

**改善点 (= research-driven)**:
- LR cosine schedule (warmup 5% + decay 5e-4 → 5e-5)
- ent_coef anneal (0.05 → 0.005)
- n_steps 1024 (= 4x、 advantage estimate 精度 ↑)
- batch_size 256 (= 4x、 gradient variance ↓)
- n_epochs 8 (= 2x、 sample reuse ↑)
- pool_max 12, save_interval 5000
- warm-start = θ.4 (= 200k step 完走結果) で進化加速

**期待**: LB 989 → 1400-1600 (= research best-case)、 ~24h on A100

**前提**: Day 4 で 200k θ.4 完走済、 Drive `/MyDrive/orbit-wars/ppo_v4_theta4.zip` 存在。
存在しなければ cell 1 で θ.3 fallback。

In [ ]:
import os
import shutil
import subprocess

from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/ykaya-jp/orbit-wars.git'
REPO_DIR = '/content/orbit-wars'
DRIVE_DIR = '/content/drive/MyDrive/orbit-wars'

# 1) Clone or update repo to latest main
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    print(f'cloned to {REPO_DIR}')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=True)
    print(f'updated {REPO_DIR}')
os.chdir(REPO_DIR)
subprocess.run(['git', 'log', '-1', '--oneline'])

# 2) Restore .gitignore-excluded files from Drive
WEIGHTS_TAR = os.path.join(DRIVE_DIR, 'orbit_wars_weights.tar.gz')
if os.path.exists(WEIGHTS_TAR):
    subprocess.run(['tar', '-xzf', WEIGHTS_TAR, '-C', REPO_DIR], check=True)
    print(f'extracted weights')

# 3) Pick warm-start: prefer θ.4 (200k), fallback to θ.3
WARM_CANDIDATES = [
    (os.path.join(DRIVE_DIR, 'ppo_v4_theta4.zip'), 'agents/proxy/ppo_v4_theta4.zip'),
    (os.path.join(DRIVE_DIR, 'ppo_v3_theta3.zip'), 'agents/proxy/ppo_v3_theta3.zip'),
]
WARM_DST = None
for src, dst_rel in WARM_CANDIDATES:
    if os.path.exists(src):
        dst = os.path.join(REPO_DIR, dst_rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if not os.path.exists(dst):
            shutil.copy(src, dst)
        WARM_DST = dst_rel
        print(f'warm-start = {dst_rel} ({os.path.getsize(src)/1024**2:.1f} MB)')
        break
if WARM_DST is None:
    raise RuntimeError('No warm-start zip found in Drive!')

# 4) Verify critical files
print('\n=== Critical files check ===')
for rel in [WARM_DST, 'agents/proxy/grid_il_lakhindar.pt', 'agents/proxy/grid_il_lakhindar.py',
            'submissions/build_konbu_topk1/main.py', 'submissions/build_konbu_topk1/weights.npz',
            'submissions/build_rudra_topk1_proper/main.py', 'tools/train_ppo_pfsp.py']:
    full = os.path.join(REPO_DIR, rel)
    print(f'  {rel}: {"OK" if os.path.exists(full) else "MISSING"}')

In [ ]:
!pip install -q --upgrade "stable-baselines3==2.8.0" "sb3-contrib==2.8.0" "gymnasium==1.2.0" "kaggle_environments==1.29.1"

import importlib, subprocess, sys
try:
    importlib.import_module('kaggle_environments.envs.orbit_wars.orbit_wars')
    print('orbit_wars env: OK (PyPI)')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps',
                    'git+https://github.com/Kaggle/kaggle-environments.git'], check=True)
    importlib.invalidate_caches()
    importlib.import_module('kaggle_environments.envs.orbit_wars.orbit_wars')
    print('orbit_wars env: OK (GitHub fallback)')

import torch, gymnasium, stable_baselines3, sb3_contrib, kaggle_environments
print(f'cuda: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')
print(f'gymnasium {gymnasium.__version__}, sb3 {stable_baselines3.__version__}, sb3-contrib {sb3_contrib.__version__}, kaggle_environments {kaggle_environments.__version__}')

In [ ]:
import os
import subprocess

REPO_DIR = '/content/orbit-wars'
os.chdir(REPO_DIR)

POOL_DIR = '/content/drive/MyDrive/orbit-wars/ppo_pfsp_pool_theta5'
os.makedirs(POOL_DIR, exist_ok=True)
OUTPUT = '/content/drive/MyDrive/orbit-wars/ppo_v5_theta5.zip'
LOG = '/content/drive/MyDrive/orbit-wars/ppo_v5_theta5_training.log'

# Day 5+ best-practice config (= research-driven、 LR cosine + ent_coef anneal)
WARM_REL = 'agents/proxy/ppo_v4_theta4.zip' if os.path.exists(os.path.join(REPO_DIR, 'agents/proxy/ppo_v4_theta4.zip')) else 'agents/proxy/ppo_v3_theta3.zip'
cmd = [
    'python', '-u', '-m', 'tools.train_ppo_pfsp',
    '--total-timesteps', '1000000',
    '--n-envs', '8',
    '--n-steps', '1024',           # 4x default (= advantage 精度)
    '--batch-size', '256',         # 4x default (= variance ↓)
    '--n-epochs', '8',             # 2x default (= sample reuse)
    '--learning-rate', '5e-4',     # peak LR (cosine 内で decay)
    '--device', 'cuda',
    '--warm-start', WARM_REL,
    '--external-opponents',
    'agents/proxy/grid_il_lakhindar.py',
    'submissions/build_konbu_topk1/main.py',
    'submissions/build_rudra_topk1_proper/main.py',
    '--pool-max', '12',            # 1.5x (= diversity ↑)
    '--save-interval', '5000',     # 2x freq (= pool 更新 ↑)
    '--pool-dir', POOL_DIR,
    '--self-play-prob', '0.6',
    '--external-prob', '0.2',
    '--reward-shaping',
    '--lr-schedule', 'cosine',     # ★ Day 5+ best-practice
    '--lr-peak', '5e-4',
    '--lr-floor', '5e-5',
    '--ent-schedule', 'anneal',    # ★ Day 5+ best-practice
    '--ent-peak', '0.05',
    '--ent-floor', '0.005',
    '--seed', '2026',
    '--output', OUTPUT,
]
print('cmd:', ' '.join(cmd))
print('---')

env = {**os.environ, 'PYTHONUNBUFFERED': '1', 'TF_CPP_MIN_LOG_LEVEL': '2'}
with open(LOG, 'w', buffering=1) as f:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
        f.write(line); f.flush()
    proc.stdout.close(); proc.wait()

print(f'\nexit: {proc.returncode}, log: {LOG}')

In [ ]:
import glob, os
OUTPUT = '/content/drive/MyDrive/orbit-wars/ppo_v5_theta5.zip'
POOL_DIR = '/content/drive/MyDrive/orbit-wars/ppo_pfsp_pool_theta5'
print('=== Output ===')
if os.path.exists(OUTPUT):
    print(f'  {OUTPUT}: {os.path.getsize(OUTPUT) / 1024**2:.1f} MB')
else:
    print(f'  MISSING: {OUTPUT}')
print('=== Pool checkpoints ===')
for p in sorted(glob.glob(f'{POOL_DIR}/*.zip')):
    print(f'  {os.path.basename(p)}: {os.path.getsize(p) / 1024**2:.1f} MB')